## Predictive Utility Asset Health Maintanence

**A Better Way to Predict Utility Asset Health and Deliver Higher Service Reliability**

Predictive maintenance uses data from sensors, such as vibration, temperature, and pressure sensors, to monitor the condition of equipment. AI algorithms can analyze this data to detect patterns and anomalies that indicate when equipment is likely to fail. By identifying potential problems early, manufacturers can schedule maintenance at a convenient time, avoiding unexpected downtime. This can help to reduce maintenance costs and increase the lifespan of equipment. In addition, predictive maintenance can also help to improve safety by identifying and addressing potential hazards before they occur.

### Predictive Utility Asset Health Maintanence as a AI for Manufacturing Usecase:

**The most promising applications of AI in manufacturing is Predictive Maintenance. It involves using AI algorithms to analyze sensor data from factory equipment to predict when maintenance is needed. This can help to prevent equipment failure and reduce downtime, increasing productivity and reducing costs.**

<video width="80%"  height="80%" 
       src=".\assets\Sensor_placed_on_the_motor_running_the_ConveyorBelt.mp4"  
       controls muted>
</video>

Video Source: https://www.linkedin.com/posts/andy-jassy-8b1615_the-orange-sensors-in-this-video-might-not-activity-7019327173962461184-CaQS?utm_source=share&utm_medium=member_desktop

## AI Project Cycle  ![AI_Project_Cycle.png](attachment:AI_Project_Cycle.png)

## Context: Understanding the Problem Statement --------Problem Scoping (AI Project Cycle - Step 1)

**An ML based predictive solution for Utility Asset Health Maintanence**

- Predictive asset maintenance solutions of huge scale typically require operating across multiple hardware architectures. Accelerating training for the ever-increasing size of datasets and machine learning models is a major challenge while adopting AI.

- Industrial scenario for predictive asset maintenance, with huge set of batch processing, requires fast prediction time without accuracy lose.

- One of the important problem statement on this industrial scenario is to improve the MLOps time for developing and deploying new models due to its ever-increasing size of datasets over a period of time. XGBoost classifier with HIST tree method has been choosen to address this problem which will improve the overall training/tuning and validation time.

![predictive_asset_maintenance_e2e_flow](assets/predictive_asset_maintenance_e2e_flow.png)

### Import the useful Packages & Libraries

We import the necessary packages and libraries to get started.                                                                                                                                                                             
Pandas - One of the most common packages used in machine learning for analysing data.                                                                                                             
Numpy - Used to deal with numerical values and to perform mathematical expressions on arrays.                   
XGBoost - Used for classification and regression problems. 

To learn more about Pandas click [here](https://pandas.pydata.org/docs/user_guide/index.html#user-guide), whereas for NumPy click [here](https://numpy.org/devdocs/user/index.html#user) and for XGBoost click [here](https://catalyst.cs.cmu.edu/projects/xgboost.html).



In [ ]:
import pandas as pd
import time
import numpy as np
import xgboost as xgb
import pickle
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler


import warnings
warnings.filterwarnings("ignore")

## Dataset:  Data Acquisition (AI Project Cycle - Step 2)

The process of gathering accurate and reliable data to work with is known as data acquisition.



Four lists are created namely manufacturer_list, species_list, district_list and treatment_list. 
A variable called *dsize* has been initialized with a value of 25000

In [ ]:
start = time.time()
dsize = 25000
print(f'Generating data with the size {dsize}')
np.random.seed(1)
manufacturer_list = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']
species_list = ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7']
district_list = ['N', 'NE', 'NW', 'E', 'W', 'S', 'SE', 'SW']
treatment_list = ['Oil', 'Pentachlorophenol', 'Untreated', 'Creosote', 'UNK', 'Cellon']

Generating data with the size 25000


The below cell generates a dataframe having 10 columns and 25000 rows.                                                                        Each column signifies a feature of the dataframe.
The *dsize* variable has been used here to set the number of rows in the dataframe.

In [ ]:
data = pd.DataFrame({"Age": np.random.choice(range(1, 101), dsize, replace=True),
                     "Elevation": np.random.randint(low=-300, high=4500, size=dsize),
                     "Pole_Height": np.random.normal(60, 15, size=dsize),
                     "Measured_Length": np.random.randint(low=1, high=2000, size=dsize),
                     "Manufacturer": np.random.choice(manufacturer_list, dsize, replace=True),
                     "Species": np.random.choice(species_list, dsize, replace=True),
                     "Number_Repairs": np.random.choice(range(1, 7), dsize, replace=True),
                     "District": np.random.choice(district_list, dsize, replace=True),
                     "Tele_Attached": np.random.choice(range(0, 2), dsize, replace=True),
                     "Original_Treatment": np.random.choice(treatment_list, dsize, replace=True)})

In [ ]:
# changing Tele_Attatched into an object variable
print('changing Tele_Attatched into an object variable')
data[['Number_Repairs', 'Tele_Attached']] = data[['Number_Repairs', 'Tele_Attached']].astype('object')

changing Tele_Attatched into an object variable


In [ ]:
# Generating our target variable Asset_Label
print('Generating our target variable Asset_Label')
data['Asset_Label'] = np.random.choice(range(0, 2), dsize, replace=True, p=[0.99, 0.01])

Generating our target variable Asset_Label


In [ ]:
# Creating correlation between our variables and our target variable<br>
# When age is 60-70 and over 95 change Asset_Label to 1
print('Creating correlation between our variables and our target variable')
print('When age is 60-70 and over 95 change Asset_Label to 1')
data['Asset_Label'] = np.where(((data.Age > 0) & (data.Age <= 5)) | (data.Age > 45),
                               1, data.Asset_Label)

Creating correlation between our variables and our target variable
When age is 60-70 and over 95 change Asset_Label to 1


In [ ]:
# When elevation is between 500-1500 change Asset_Label to 1
print('When elevation is between 500-1500 change Asset_Label to 1')
data['Asset_Label'] = np.where((data.Elevation >= -300) & (data.Elevation <= 1400),
                               1, data.Asset_Label)

When elevation is between 500-1500 change Asset_Label to 1


In [ ]:
# When Manufacturer is A, E, or H change Asset_Label to have  95% 0's
print("When Manufacturer is A, E, or H change Asset_Label to have  95% 0's")
data['Temp_Var'] = np.random.choice(range(0, 2), dsize, replace=True, p=[0.95, 0.05])
data['Asset_Label'] = np.where((data.Manufacturer == 'A') |
                               (data.Manufacturer == 'E') |
                               (data.Manufacturer == 'H'),
                               data.Temp_Var,
                               data.Asset_Label)

When Manufacturer is A, E, or H change Asset_Label to have  95% 0's


In [ ]:
# When Species is C2 or C5 change Asset_Label to have 90% to 0's
print("When Species is C2 or C5 change Asset_Label to have 90% to 0's")
data['Temp_Var'] = np.random.choice(range(0, 2), dsize, replace=True, p=[0.9, 0.1])
data['Asset_Label'] = np.where((data.Species == 'C2') | (data.Species == 'C5'),
                               data.Temp_Var,
                               data.Asset_Label)

When Species is C2 or C5 change Asset_Label to have 90% to 0's


In [ ]:
# When District is NE or W change Asset_Label to have 90% to 0's
print("When District is NE or W change Asset_Label to have 90% to 0's")
data['Temp_Var'] = np.random.choice(range(0, 2), dsize, replace=True, p=[0.95, 0.05])
data['Asset_Label'] = np.where((data.District == 'NE') | (data.District == 'W'),
                               data.Temp_Var, data.Asset_Label)

When District is NE or W change Asset_Label to have 90% to 0's


In [ ]:
# When District is Untreated change Asset_Label to have 70% to 1's
print("When District is Untreated change Asset_Label to have 70% to 1's")
data['Temp_Var'] = np.random.choice(range(0, 2), dsize, replace=True, p=[0.25, 0.75])
data['Asset_Label'] = np.where((data.Original_Treatment == 'Untreated'),
                               data.Temp_Var,
                               data.Asset_Label)

When District is Untreated change Asset_Label to have 70% to 1's


In [ ]:
# When Age is greater than 90 and Elevaation is less than 1200
# and Original_treatment is Oil change Asset_Label to have 90% to 1's
print("When Age is greater than 90 and Elevaation is less than 1200 \nand Original_treatment is Oil change Asset_Label to have 90% to 1's")
data['Temp_Var'] = np.random.choice(range(0, 2), dsize, replace=True, p=[0.05, 0.95])
data['Asset_Label'] = np.where((data.Age >= 20) &
                               (data.Elevation <= 1000) &
                               (data.Original_Treatment == 'Oil') &
                               (data.Pole_Height >= 60),
                               data.Temp_Var,
                               data.Asset_Label)

When Age is greater than 90 and Elevaation is less than 1200 
and Original_treatment is Oil change Asset_Label to have 90% to 1's


In [ ]:
#The label'Temp_Var' is being removed using the drop function.
data.drop('Temp_Var', axis=1, inplace=True)

In [ ]:
Categorical_Variables = pd.get_dummies(
                           data[[
                               'Manufacturer',
                               'District',
                               'Species',
                               'Original_Treatment']],
                           drop_first=True)
data = pd.concat([data, Categorical_Variables], axis=1)
data.drop(['Manufacturer', 'District', 'Species', 'Original_Treatment'], axis=1, inplace=True)

data = data.astype({'Tele_Attached': 'int32', 'Number_Repairs': 'float64'})

etime = time.time() - start
datasize = data.shape
print(f"=====> Time taken {etime} secs for data generation for the size of {datasize}")

=====> Time taken 0.28171372413635254 secs for data generation for the size of (25000, 34)


### View the data

In [ ]:
#Prints the first five entry of the dataframe
data.head()

,Age,Elevation,Pole_Height,Measured_Length,Number_Repairs,Tele_Attached,Asset_Label,Manufacturer_B,Manufacturer_C,Manufacturer_D,...,Species_C3,Species_C4,Species_C5,Species_C6,Species_C7,Original_Treatment_Creosote,Original_Treatment_Oil,Original_Treatment_Pentachlorophenol,Original_Treatment_UNK,Original_Treatment_Untreated
0,38,2076,61.067093,1355,5.0,1,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0
1,13,4275,65.830643,1749,4.0,0,1,0,0,1,...,0,0,1,0,0,0,1,0,0,0
2,73,4241,71.412853,597,4.0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,10,2226,57.331145,896,6.0,1,0,0,0,0,...,0,1,0,0,0,0,1,0,0,0
4,76,2773,58.442710,315,5.0,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0


## Data Preprocessing ------ Data Exploration(AI Project Cycle - Step 3)

### View the information of data

Data exploration is performed to gather information on data and based on the gaind insights, various pre-processing functions are implemented. 

The general insights that are obtained from data exploration are                                                                                               
1.Statistical information of the entire data                                                                                             
2.Outliers present in the data                                                                                                                              
3.Datatypes                                                                                                                                                                  
4.Null values

In [ ]:
# info() helps summarize the dataset- It gives basic information like number of non-null values, datatypes and memory usage
# It is a good practise to start by this information
data.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 34 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Age                                   25000 non-null  int32  
 1   Elevation                             25000 non-null  int32  
 2   Pole_Height                           25000 non-null  float64
 3   Measured_Length                       25000 non-null  int32  
 4   Number_Repairs                        25000 non-null  float64
 5   Tele_Attached                         25000 non-null  int32  
 6   Asset_Label                           25000 non-null  int32  
 7   Manufacturer_B                        25000 non-null  uint8  
 8   Manufacturer_C                        25000 non-null  uint8  
 9   Manufacturer_D                        25000 non-null  uint8  
 10  Manufacturer_E                        25000 non-null  uint8  
 11  Manufacturer_F 

The describe() function gives the numerical statistical information of the dataframe    

count - The number of non-empty values.                                                                                   
mean - The average value                                                                                               
std - The standard deviation                                                                                    
min - the minimum value                                                                                     
25% - The 25% percentile                                                                                    
50% - The 50% percentile                                                                                  
75% - The 75% percentile                                                                    
max - the maximum value 

In [ ]:
data.describe()

,Age,Elevation,Pole_Height,Measured_Length,Number_Repairs,Tele_Attached,Asset_Label,Manufacturer_B,Manufacturer_C,Manufacturer_D,...,Species_C3,Species_C4,Species_C5,Species_C6,Species_C7,Original_Treatment_Creosote,Original_Treatment_Oil,Original_Treatment_Pentachlorophenol,Original_Treatment_UNK,Original_Treatment_Untreated
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,...,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.00000
mean,50.529080,2097.143560,60.030777,992.855640,3.496680,0.501200,0.397440,0.099160,0.100280,0.096480,...,0.143640,0.146400,0.142280,0.141800,0.141080,0.168440,0.163880,0.168600,0.169880,0.16124
std,28.785265,1381.601431,15.039251,577.387953,1.703674,0.500009,0.489378,0.298883,0.300379,0.295254,...,0.350731,0.353514,0.349344,0.348852,0.348111,0.374264,0.370174,0.374406,0.375535,0.36776
min,1.000000,-300.000000,6.592793,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,25.000000,903.000000,49.870047,491.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,51.000000,2089.000000,60.171125,989.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,75.000000,3290.000000,70.159988,1495.000000,5.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,100.000000,4499.000000,120.489768,1999.000000,6.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


### Find the total number of missing values feature wise

In [ ]:
#It returns the number of null values in the dataframe for every column feature
#Using info() we can view the number of non-null values whereas isnull() gives the number of null values
data.isnull().sum()

Age                                     0
Elevation                               0
Pole_Height                             0
Measured_Length                         0
Number_Repairs                          0
Tele_Attached                           0
Asset_Label                             0
Manufacturer_B                          0
Manufacturer_C                          0
Manufacturer_D                          0
Manufacturer_E                          0
Manufacturer_F                          0
Manufacturer_G                          0
Manufacturer_H                          0
Manufacturer_I                          0
Manufacturer_J                          0
District_N                              0
District_NE                             0
District_NW                             0
District_S                              0
District_SE                             0
District_SW                             0
District_W                              0
Species_C2                        

### Splitting dataset into separate training and test set 
The sklearn train_test_split method has been used to split the dataset into train and test cohorts.                                                                                                                                                        Here, 25% of the data has been allocated as test data.                                                                        Since, Asset_Label has been removed from the overall data(X) and y has only one feature which is Asset_Label as the target variable. 

In [ ]:
datasize = data.shape

start = time.time()
X = data.drop('Asset_Label', axis=1)
y = data.Asset_Label

# original split .25
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25)

df_num_train = X_train.select_dtypes(['float', 'int', 'int32'])
df_num_test = X_test.select_dtypes(['float', 'int', 'int32'])

### Scaling the data
 
We are performing scaling here to normalize the data within a sepcific range.                                                                              
RobustScaler() uses interquartile range(IQR) and median is used to scale the data.

to know more about Roburst Scaler click [here](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html)

In [ ]:
robust_scaler = RobustScaler()
X_train_scaled = robust_scaler.fit_transform(df_num_train)
X_test_scaled = robust_scaler.transform(df_num_test)

In [ ]:
# Making them pandas dataframes
X_train_scaled_transformed = pd.DataFrame(X_train_scaled,
                                          index=df_num_train.index,
                                          columns=df_num_train.columns)
X_test_scaled_transformed = pd.DataFrame(X_test_scaled,
                                         index=df_num_test.index,
                                         columns=df_num_test.columns)

del X_train_scaled_transformed['Number_Repairs']
del X_train_scaled_transformed['Tele_Attached']

del X_test_scaled_transformed['Number_Repairs']
del X_test_scaled_transformed['Tele_Attached']

In [ ]:
# Dropping the unscaled numerical columns
X_train = X_train.drop(['Age', 'Elevation', 'Pole_Height', 'Measured_Length'], axis=1)
X_test = X_test.drop(['Age', 'Elevation', 'Pole_Height', 'Measured_Length'], axis=1)

In [ ]:
# Creating train and test data with scaled numerical columns
X_train_scaled_transformed = pd.concat([X_train_scaled_transformed, X_train], axis=1)
X_test_scaled_transformed = pd.concat([X_test_scaled_transformed, X_test], axis=1)

In [ ]:
#len(dataframe)returns the no of rows in the dataframe
len(X_train_scaled_transformed)

18750

In [ ]:
len(X_test_scaled_transformed)

6250

In [ ]:
#Prints the first five entry of the dataframe
X_train_scaled_transformed.head()

,Age,Elevation,Pole_Height,Measured_Length,Number_Repairs,Tele_Attached,Manufacturer_B,Manufacturer_C,Manufacturer_D,Manufacturer_E,...,Species_C3,Species_C4,Species_C5,Species_C6,Species_C7,Original_Treatment_Creosote,Original_Treatment_Oil,Original_Treatment_Pentachlorophenol,Original_Treatment_UNK,Original_Treatment_Untreated
22708,0.88,0.089599,-0.128359,0.900424,1.0,1,1,0,0,0,...,0,0,1,0,0,1,0,0,0,0
3696,-0.12,0.023601,1.112967,0.804592,4.0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0
21328,0.64,0.000209,1.297488,0.146743,1.0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,1
2538,0.66,0.837719,0.497244,0.090841,4.0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
17784,0.80,0.234962,0.827295,0.023958,4.0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0


In [ ]:
#Prints the first five entry of the dataframe
X_test_scaled_transformed.head()

,Age,Elevation,Pole_Height,Measured_Length,Number_Repairs,Tele_Attached,Manufacturer_B,Manufacturer_C,Manufacturer_D,Manufacturer_E,...,Species_C3,Species_C4,Species_C5,Species_C6,Species_C7,Original_Treatment_Creosote,Original_Treatment_Oil,Original_Treatment_Pentachlorophenol,Original_Treatment_UNK,Original_Treatment_Untreated
9850,0.58,0.458438,-1.538635,-0.882456,6.0,0,0,0,0,0,...,0,0,1,0,0,1,0,0,0,0
7126,0.62,0.233292,0.651497,0.642875,5.0,1,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0
198,-0.06,0.253342,1.852966,0.442226,1.0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
23011,-0.92,0.486842,0.300092,0.082855,3.0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
20033,-0.14,0.921679,0.278658,-0.951335,3.0,1,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0


In [ ]:
y_train.head()

22708    0
3696     0
21328    1
2538     0
17784    0
Name: Asset_Label, dtype: int32

## Building a Model ------ Modeling ( AI Project Cycle - Step 4)

We will now use the XGBoost library to build the model. We will begin by defining the hyperparameters to make the ensemble model robust.

A powerful machine-learning algorithm called XGBoost can assist you in comprehending your data and taking wiser decisions.

An application of gradient-boosting decision trees is XGBoost. Data scientists and researchers from all around the world have utilised it to enhance their machine-learning models.

To know more about XGBoost click [here](https://xgboost.readthedocs.io/en/stable/python/python_api.html)

In [ ]:
cv_val = 2
tuning = 1
TUNING = False
if tuning == "1":
    TUNING = True

def fit_xgb_model(x_data, y_data):
    """Use a XGBClassifier for this problem."""
    # prepare data for xgboost training
    dtrain = xgb.DMatrix(x_data, y_data)
    label = dtrain.get_label()
    ratio = float(np.sum(label == 0)) / np.sum(label == 1)
    # Set xgboost parameters
    parameters = {'scale_pos_weight': ratio.round(2), 'tree_method': 'hist'}

    # define the model to use
    if TUNING is False:
        xg_cl = xgb.XGBClassifier(use_label_encoder=False)
    else:
        xg_cl = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    xg_cl.set_params(**parameters)
    # Train the model
    xg_cl.fit(x_data, y_data)
    return xg_cl

In [ ]:
X_train_scaled_transformed = X_train_scaled_transformed.astype(
                                {'Tele_Attached': 'float64',
                                 'Number_Repairs': 'float64'})
X_test_scaled_transformed = X_test_scaled_transformed.astype(
                                {'Tele_Attached': 'float64',
                                 'Number_Repairs': 'float64'})

**What are Hyperparameters?**
•	A hyperparameter is a parameter whose value is set before the learning process begins

•	Algorithm hyperparameters affect the speed and quality of the learning process

•	Hyperparameters are important because they can have a direct impact on the behaviour of the training algorithm and have a   
    significant impact on the performance of the model being trained
    
Examples - Number of epochs, number of clusters, batch size etc

**Performing hyperparameter tuning**

As shown below, we are trying to find which algorithm should be used to build the model and which hyperparameters would contribute to better performance:

In this case, 
**C**(adds penalty to each misclassified point) and kernel are the hyperparameters of SVM

**n_estimators**(this specifies the number of trees you want to build before taking the maximum voting or averages of predictions. Higher number of trees give you better performance but makes your code slower) - hyperparameter for random forest

**C**(logistic regression) large values of C give more freedom to the model. Conversely, smaller values of C constrain the model more

### Training the Ensemble Model

XGBoost model has been fitted and the training time is checked.                              
Random search is used to select the best possible set of hyperparameters by creating a dictionary called '*params*'.

In [ ]:
# Training
tstart = time.time()
xgb_model = fit_xgb_model(X_train_scaled_transformed, y_train)
ttime = time.time() - tstart
if TUNING is True:
    # GridSearchCV
    def timer(start_time=None):  # pylint: disable=missing-function-docstring
        if not start_time:
            start_time = datetime.now()
            return start_time
        if start_time:
            thour, temp_sec = divmod((datetime.now() - start_time).total_seconds(), 3600)
            tmin, tsec = divmod(temp_sec, 60)
            print(f'Time taken: {thour} hours {tmin} minutes and {round(tsec, 2)} seconds.')
        return 0

    # Hyper parameters for tuning
    params = {
        'min_child_weight': [1, 5, 10],
        'gamma': [0.5, 1, 1.5, 2, 5],
        # 'subsample': [0.6, 0.8, 1.0],
        # 'colsample_bytree': [0.6, 0.8, 1.0],
        'max_depth': [3, 4, 5],
        # 'learning_rate': [0.001, 0.01]
        }

    # n_jobs should be used according to the underlying HW accelerators, hence -1 is given
    random_search = GridSearchCV(xgb_model, param_grid=params, cv=cv_val, verbose=10, n_jobs=-1)

    # Here we go
    starttime = timer(None)  # timing starts from this point for "starttime" variable
    random_search.fit(X_train_scaled_transformed, y_train)
    timer(starttime)

## Evaluating the Model ------- Evaluation (AI Project Cycle - Step 5)

By feeding the test dataset into the model and comparing the outputs with actual solutions, evaluation is a procedure for determining the dependability of any AI model based on outputs.

### Testing the Ensemble Model

In [ ]:
# XGBoost vanilla prediction (for accuracy comparison)
dtest = xgb.DMatrix(X_test_scaled_transformed, y_test)
pstart = time.time()
xgb_prediction = xgb_model.predict(X_test_scaled_transformed)
ptime = time.time() - pstart
xgb_errors_count = np.count_nonzero(xgb_prediction - np.ravel(y_test))

xgb_total = ptime

y_test = np.ravel(y_test)

etime = time.time() - start
accuracy_scr = 1 - xgb_errors_count / xgb_prediction.shape[0]
print(f'=====> Time taken {etime} secs for training and prediction for the data size of {datasize}')
print(f'=====> Training Time {ttime} secs')
print(f'=====> Prediction Time {ptime} secs')
print(f'=====> XGBoost accuracy score {accuracy_scr}')


print(f'\nOn predicting the test data using XGBoost algorithm, an accuaracy of {accuracy_scr*100}% and a training time of {ttime} seconds has been achieved. ')

=====> Time taken 0.6115851402282715 secs for training and prediction for the data size of (25000, 34)
=====> Training Time 0.32915711402893066 secs
=====> Prediction Time 0.014960289001464844 secs
=====> XGBoost accuracy score 0.92032

On predicting the test data using XGBoost algorithm, an accuaracy of 92.032% and a training time of 0.32915711402893066 seconds has been achieved. 
